### TODO

### 本日進度  
* 整合[股票分析機器人](https://colab.research.google.com/drive/1gneFQWNI6PfkuE9WvN26bVMm702weH8D#scrollTo=j1x0glPsNJe2)：[最強 AI 投資分析：打造自己的股市顧問機器人，股票趨勢分析×年報解讀×選股推薦×風險管理](https://www.books.com.tw/products/0010973679)（[F3933 最強 AI 投資分析 - 服務專區](https://www.flag.com.tw/bk/t/f3933)）

##### 1️⃣ 匯入套件

In [1]:
from openai import OpenAI, OpenAIError # 串接 OpenAI API
import yfinance as yf
import pandas as pd # 資料處理套件
import numpy as np
import datetime as dt # 時間套件
import requests
from bs4 import BeautifulSoup

##### 2️⃣ 輸入 OpenAI API KEY

In [2]:
import getpass # 保密輸入套件
api_key = getpass.getpass("請輸入金鑰：")
client = OpenAI(api_key = api_key) # 建立 OpenAI 物件

請輸入金鑰： ········


##### 3️⃣ 取得股價資料

In [3]:
# 從 yfinance 取得一周股價資料
def stock_price(stock_id="大盤", days = 10):
    if stock_id == "大盤":
        stock_id="^TWII"
    else:
        stock_id += ".TW"

    end = dt.date.today() # 資料結束時間
    start = end - dt.timedelta(days=days) # 資料開始時間
    # 下載資料
    df = yf.download(stock_id, start=start)

    # 更換列名
    df.columns = ['開盤價', '最高價', '最低價',
                    '收盤價', '調整後收盤價', '成交量']

    data = {
        '日期': df.index.strftime('%Y-%m-%d').tolist(),
        '收盤價': df['收盤價'].tolist(),
        '每日報酬': df['收盤價'].pct_change().tolist(),
        '漲跌價差': df['調整後收盤價'].diff().tolist()
    }

    return data

print(stock_price("2330"))

[*********************100%%**********************]  1 of 1 completed
{'日期': ['2023-12-27', '2023-12-28', '2023-12-29', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'], '收盤價': [592.0, 593.0, 593.0, 593.0, 578.0, 580.0, 576.0], '每日報酬': [nan, 0.0016891891891892552, 0.0, 0.0, -0.025295109612141653, 0.0034602076124568004, -0.006896551724137945], '漲跌價差': [nan, 1.0, 0.0, 0.0, -15.0, 2.0, -4.0]}


##### 4️⃣ 取得基本面資料

In [4]:
# 基本面資料
def stock_fundamental(stock_id= "大盤"):
    if stock_id == "大盤":
        return None

    stock_id += ".TW"
    stock = yf.Ticker(stock_id)

    # 營收成長率
    quarterly_revenue_growth = np.round(stock.quarterly_financials.loc["Total Revenue"].pct_change(-1).dropna().tolist(), 2)

    # 每季EPS
    quarterly_eps = np.round(stock.quarterly_financials.loc["Basic EPS"].dropna().tolist(), 2)

    # EPS季增率
    quarterly_eps_growth = np.round(stock.quarterly_financials.loc["Basic EPS"].pct_change(-1).dropna().tolist(), 2)

    # 轉換日期
    dates = [date.strftime('%Y-%m-%d') for date in stock.quarterly_financials.columns]

    data = {
        '季日期': dates[:len(quarterly_revenue_growth)],  # 以最短的數據列表長度為准，確保數據對齊
        '營收成長率': quarterly_revenue_growth.tolist(),
        'EPS': quarterly_eps.tolist(),
        'EPS 季增率': quarterly_eps_growth.tolist()
    }

    return data

print(stock_fundamental("2330"))

{'季日期': ['2023-09-30', '2023-06-30', '2023-03-31'], '營收成長率': [1136.03, -0.05, -0.19], 'EPS': [8.14, 7.01, 7.98, 11.41], 'EPS 季增率': [0.16, -0.12, -0.3]}


##### 5️⃣ 取得新聞資料

In [5]:
# 新聞資料
def stock_news(stock_name ="大盤"):
    if stock_name == "大盤":
        stock_name="台股 -盤中速報"

    data=[]
    # 取得 Json 格式資料
    json_data = requests.get(f'https://ess.api.cnyes.com/ess/api/v1/news/keyword?q={stock_name}&limit=5&page=1').json()

    # 依照格式擷取資料
    items=json_data['data']['items']
    for item in items:
        # 網址、標題和日期
        news_id = item["newsId"]
        title = item["title"]
        publish_at = item["publishAt"]
        # 使用 UTC 時間格式
        utc_time = dt.datetime.utcfromtimestamp(publish_at)
        formatted_date = utc_time.strftime('%Y-%m-%d')
        # 前往網址擷取內容
        url = requests.get(f'https://news.cnyes.com/'
                           f'news/id/{news_id}').content
        soup = BeautifulSoup(url, 'html.parser')
        p_elements=soup .find_all('p')
        # 提取段落内容
        p=''
        for paragraph in p_elements[4:]:
            p+=paragraph.get_text()
        data.append([stock_name, formatted_date ,title,p])
    return data

print(stock_news("台積電"))

[['台積電', '2024-01-06', '鉅亨速報 - Factset 最新調查：創意(3443-TW)EPS預估下修至34.5元，預估目標價為1662.5元', '根據FactSet最新調查，共19位分析師，對創意(3443-TW)做出2024年EPS預估：中位數由34.94元下修至34.5元，其中最高估值40.84元，最低估值28.99元，預估目標價為1662.5元。市場預估EPS市場預估營收歷史獲利表現詳細資訊請看台股內頁：https://www.cnyes.com/twstock/3443/financials/ratios最新相關新聞資料來源：Factset，數據僅供參考，不作為投資建議。'], ['台積電', '2024-01-06', '〈熱門股〉智原傳獲5奈米設計訂單 逆勢周漲8%', '台股本周 2024 年開局重挫 411 點，智原 (3035-TW) 則因傳出接獲中國自駕車業者 Momenta 的 5 奈米設計訂單，吸引市場積極卡位，股價逆勢上漲，單周強漲 7.56%，收在 384 元，企圖挑戰前高 391 元。法人指出，智原該客戶的自駕晶片設計訂單，將導入台積電 5 奈米製程量產，預計實際量產時間落在 2025 年下半年，實際營收貢獻仍小。另外，智原本周也正式宣布成為 Arm Total Design 的設計服務合作夥伴，為台灣唯一一家，未來將善用在 Arm Neoverse 計算子系統 (CSS) 的優勢，提供客戶先進雲端、高效能運算 (HPC) 和人工智慧 (AI) 晶片的設計服務。智原本周在均線多頭排列下，股價穩步上攻，週五終場收在 384 元，寫近一個月來新高，成交量也放大至 7.7 萬張，三大法人僅自營商逢高調節 589 張，外資狂買 1.3 萬張，投信也加碼 2589 張。\xa0'], ['台積電', '2024-01-06', 'SCFI連6紅、日本能登半島強震、第5檔千億台股ETF誕生 本周大事回顧', '最新一期 SCFI 續揚，呈現連 6 紅，紅海危機造成的缺艙潮延續中；日本能登半島元旦發生芮氏規模 7.6 強震，關注半導體業供應鏈是否受影響；第 5 檔台股千億 ETF 誕生，群益台灣精選高息 (00919-TW) 突破千億元大關，以下為本周大事回顧：\xa0上海航交所周五 (5 日) 公告最新一期上海出口集裝箱運

##### 6️⃣ 爬取股號、股名對照表

In [6]:
# 取得全部股票的股號、股名
def stock_name():
      print("線上讀取股號、股名、及產業別")

      response = requests.get('https://isin.twse.com.tw/isin/C_public.jsp?strMode=2')
      url_data = BeautifulSoup(response.text, 'html.parser')
      stock_company = url_data.find_all('tr')

      # 資料處理
      data = [
          (row.find_all('td')[0].text.split('\u3000')[0].strip(),
           row.find_all('td')[0].text.split('\u3000')[1],
           row.find_all('td')[4].text.strip())
          for row in stock_company[2:] if len(row.find_all('td')[0].text.split('\u3000')[0].strip()) == 4
      ]

      df = pd.DataFrame(data, columns=['股號', '股名', '產業別'])

      return df

name_df = stock_name()

線上讀取股號、股名、及產業別


##### 7️⃣ 取得股票名稱

In [7]:
# 取得股票名稱
def get_stock_name(stock_id, name_df):
    return name_df.set_index('股號').loc[stock_id, '股名']

print(name_df.head())
print("--------------------------")
print(get_stock_name("2501",name_df))

     股號  股名   產業別
0  1101  台泥  水泥工業
1  1102  亞泥  水泥工業
2  1103  嘉泥  水泥工業
3  1104  環泥  水泥工業
4  1108  幸福  水泥工業
--------------------------
國建


##### 8️⃣ 建構 GPT 3.5 模型

In [8]:
# 建立 GPT 3.5-16k 模型
def get_reply(messages):
    try:
        response = client.chat.completions.create(
            model = "gpt-3.5-turbo-1106",
            messages = messages
        )
        reply = response.choices[0].message.content
    except OpenAIError as err:
        reply = f"發生 {err.type} 錯誤\n{err.message}"
    return reply

# 建立訊息指令(Prompt)
def generate_content_msg(stock_id, name_df):

    stock_name = get_stock_name(
        stock_id, name_df) if stock_id != "大盤" else stock_id

    price_data = stock_price(stock_id)
    news_data = stock_news(stock_name)

    content_msg = f'請依據以下資料來進行分析並給出一份完整的分析報告:\n'

    content_msg += f'近期價格資訊:\n {price_data}\n'

    if stock_id != "大盤":
        stock_value_data = stock_fundamental(stock_id)
        content_msg += f'每季營收資訊：\n {stock_value_data}\n'

    content_msg += f'近期新聞資訊: \n {news_data}\n'
    content_msg += f'請給我{stock_name}近期的趨勢報告,請以詳細、\
      嚴謹及專業的角度撰寫此報告,並提及重要的數字, reply in 繁體中文'

    return content_msg

# StockGPT
def stock_gpt(stock_id, name_df=name_df):
    content_msg = generate_content_msg(stock_id, name_df)

    msg = [{
        "role": "system",
        "content": f"你現在是一位專業的證券分析師, 你會統整近期的股價\
      、基本面、新聞資訊等方面並進行分析, 然後生成一份專業的趨勢分析報告"
    }, {
        "role": "user",
        "content": content_msg
    }]

    reply_data = get_reply(msg)

    return reply_data


##### 9️⃣ 大盤趨勢報告

In [9]:
reply = stock_gpt(stock_id="大盤")
print(reply)

[*********************100%%**********************]  1 of 1 completed
大盤近期趨勢分析報告：

本周台股整體呈現震盪調整的走勢。在收盤價方面，台股於2023年12月27日至2024年1月5日的收盤價依序為17891.5、17910.37、17930.81、17853.76、17559.31、17549.65、17519.14。從每日收盤價來看，市場整體呈現下跌的趨勢。

在新聞資訊方面，值得關注的重要事件包括：SCFI連6紅、日本能登半島強震、第5檔千億台股ETF誕生、外資2023年淨匯入逾9100億元創史上新高等。外資於2023年累計淨匯入金額達到約9138億元，刷新歷史新高，顯示台股受到熱錢追捧。值得一提的是，日本能登半島元旦發生芮氏規模 7.6 強震，以及CES展登場，這些事件可能對台股的行情產生一定的影響。

從收盤價和新聞資訊來看，整體趨勢是市場呈現調整的態勢。外資匯入創歷史新高，但市場整體上呈現震盪調整，部分股票受到特定事件的影響有所波動。投資者應密切關注市場動向，並根據自身風險承受能力和投資策略來做出相應的操作。

以上報告僅供參考，投資者還需進一步分析市場動向和該股票的基本面資料，以制定適合自己的投資策略。


##### 🔟 個股分析報告

In [10]:
reply = stock_gpt(stock_id="2330")
print(reply)

[*********************100%%**********************]  1 of 1 completed
台積電（2330-TW）近期股價走勢呈現複雜性，我們將利用近期的價格資訊、基本面和新聞資訊進行詳細分析。

首先，我們觀察到從 2023 年 12 月 27 日至 2024 年 1 月 5 日的股價走勢。從這段時間的每日報酬率來看，台積電股價呈現波動明顯，期間累積的價格波動幅度較大。從收盤價來看，最高為 593.0 元，最低為 576.0 元。這段時間的股價波動明顯，可能受到市場消息的影響。需要密切關注價格波動的原因及未來可能的趨勢。

其次，我們檢視台積電最近的每季營收資訊。從 2023 年 9 月 30 日至 2023 年 3 月 31 日，公司的營收成長率分別為 1136.03%、-0.05%、-0.19%。這顯示公司處於一個高度成長的階段，然而也存在營收下滑的風險。同時，我們觀察到最近的 EPS 季增率分別為 0.16、-0.12、-0.3，顯示公司獲利表現也出現波動。需要進一步分析公司的營收結構和利潤狀況。

最後，我們關注最近的新聞資訊。根據新聞資訊，分析師對於台積電的未來表現給出了不同的預測。另外，有關台積電與合作夥伴的消息也可能對公司股價造成影響。另外，供應鏈消息顯示，蘋果將持續增加對台積電 3 奈米製程晶片的採用，對公司未來的訂單和產能利用率有著重大影響。

綜合以上資訊，我們認為台積電股價波動較大，存在一定的風險，需要密切關注市場動態。同時，需要關注公司的營收結構和利潤狀況，以及與重要合作夥伴的合作情況。建議投資者在考慮投資台積電時要謹慎行事，並密切關注後續市場變化。
